# LLM Evaluation — With Description

Predict loan outcomes using an LLM with **structured features + borrower description**.
Compare results against the XGBoost model and the no-description LLM baseline on the same 100 test samples.

## Setup

In [1]:
import sys
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

from llm_utils import (
    load_llm_sample, run_ml_on_sample,
    build_system_prompt, build_user_prompt,
    call_llm, parse_llm_response,
    evaluate_predictions, compare_results,
    RESULTS_DIR
)

In [2]:
# ── Configuration ──────────────────────────────────────────────────────────
# API key loaded automatically from .env file
API_PROVIDER = "gemini"            # "gemini", "anthropic", or "openai"
MODEL_NAME   = "gemini-2.5-flash"  # None = use default for provider
API_KEY      = None                # None = read from .env / environment

## Load Data & Run XGBoost

In [3]:
llm_sample = load_llm_sample()
y_true = llm_sample['loan_status'].values

print(f"Sample size: {len(llm_sample)}")
print(f"Class distribution:\n{llm_sample['loan_status'].value_counts()}")

Sample size: 100
Class distribution:
loan_status
1    84
0    16
Name: count, dtype: int64


In [4]:
xgb_probs, xgb_preds = run_ml_on_sample(llm_sample)
print(f"XGBoost predictions ready: {len(xgb_preds)} samples")

XGBoost predictions ready: 100 samples


## LLM Predictions (With Description)

In [5]:
system_prompt = build_system_prompt()

# Preview the prompt for the first loan (now includes description)
sample_prompt = build_user_prompt(llm_sample.iloc[0], include_desc=True)
print("System prompt:")
print(system_prompt)
print("\n" + "=" * 50)
print("\nSample user prompt (with description):")
print(sample_prompt)

System prompt:
You are a credit risk analyst. Given a loan application's features, predict whether the borrower will fully repay the loan or default (charge off).

Respond ONLY with valid JSON in this exact format:
{"prediction": <1 or 0>, "reasoning": "<brief explanation>"}

Where:
- prediction: 1 = Fully Paid, 0 = Charged Off
- reasoning: 1-2 sentence explanation of your prediction


Sample user prompt (with description):
Predict the outcome for this loan application:

- Loan amount requested ($): 10000.0
- Loan term:  36 months
- Interest rate (%): 12.12
- Monthly payment ($): 332.72
- LC assigned loan grade: B
- LC assigned loan sub-grade: B3
- Home ownership status: RENT
- Annual income ($): 82000.0
- Income verification status: Not Verified
- Stated loan purpose: credit_card
- Debt-to-income ratio: 3.57
- Earliest credit line date: 1983-09-01
- Number of open credit accounts: 6.0
- Has derogatory public records (0/1): 0
- Revolving balance ($): 6990.0
- Revolving utilization rate

In [6]:
# Run LLM on all 100 samples WITH description (resumable)
if 'llm_predictions' not in dir() or not llm_predictions:
    llm_predictions = []
    llm_reasonings = []
    llm_raw_responses = []

start_from = len(llm_predictions)
if start_from > 0:
    print(f"Resuming from sample {start_from}/{len(llm_sample)}")

for i, (_, row) in enumerate(tqdm(llm_sample.iterrows(), total=len(llm_sample))):
    if i < start_from:
        continue

    user_prompt = build_user_prompt(row, include_desc=True)

    raw = call_llm(
        system_prompt, user_prompt,
        api_provider=API_PROVIDER, model=MODEL_NAME, api_key=API_KEY
    )
    llm_raw_responses.append(raw)

    parsed = parse_llm_response(raw)
    llm_predictions.append(parsed['prediction'])
    llm_reasonings.append(parsed['reasoning'])

print(f"\nCompleted: {len(llm_predictions)} predictions")
print(f"Parse errors: {sum(1 for p in llm_predictions if p is None)}")

100%|██████████| 100/100 [13:09<00:00,  7.90s/it]


Completed: 100 predictions
Parse errors: 0


## Evaluation

In [7]:
llm_metrics = evaluate_predictions(y_true, llm_predictions, label="LLM (With Desc)")
xgb_metrics = evaluate_predictions(y_true, xgb_preds.tolist(), label="XGBoost")


LLM (With Desc) Results (100 samples)
Accuracy: 63.0%

Classification Report:
              precision    recall  f1-score   support

 Charged Off       0.23      0.56      0.33        16
  Fully Paid       0.89      0.64      0.74        84

    accuracy                           0.63       100
   macro avg       0.56      0.60      0.54       100
weighted avg       0.78      0.63      0.68       100

Confusion Matrix:
[[ 9  7]
 [30 54]]

XGBoost Results (100 samples)
Accuracy: 71.0%

Classification Report:
              precision    recall  f1-score   support

 Charged Off       0.29      0.56      0.38        16
  Fully Paid       0.90      0.74      0.81        84

    accuracy                           0.71       100
   macro avg       0.59      0.65      0.60       100
weighted avg       0.80      0.71      0.74       100

Confusion Matrix:
[[ 9  7]
 [22 62]]


In [8]:
comparison = compare_results(y_true, llm_predictions, xgb_preds.tolist(), llm_reasonings)

print(f"LLM accuracy: {comparison['llm_correct'].mean()*100:.1f}%")
print(f"XGBoost accuracy: {comparison['xgb_correct'].mean()*100:.1f}%")
print(f"\nAgreement between LLM and XGBoost: {(comparison['llm_pred'] == comparison['xgb_pred']).mean()*100:.1f}%")

comparison.head(10)

LLM accuracy: 63.0%
XGBoost accuracy: 71.0%

Agreement between LLM and XGBoost: 84.0%


,actual,llm_pred,xgb_pred,llm_correct,xgb_correct,llm_reasoning
0,1,1,1,1,1,The borrower exhibits a very low debt-to-incom...
1,1,0,0,0,0,The borrower presents significant risk factors...
2,1,0,0,0,0,Despite a low debt-to-income ratio and excelle...
3,1,1,0,1,0,"Despite high revolving utilization, the borrow..."
4,1,0,1,0,1,The borrower exhibits significant financial st...
5,1,0,0,0,0,The borrower has a high-risk D-grade loan with...
6,1,1,1,1,1,The borrower exhibits a strong financial profi...
7,1,1,1,1,1,The borrower demonstrates strong creditworthin...
8,1,0,1,0,1,Despite a long credit history and no prior der...
9,1,1,1,1,1,The borrower has an exceptionally long credit ...


In [9]:
# Cases where LLM and XGBoost disagree
disagree = comparison[comparison['llm_pred'] != comparison['xgb_pred']]
print(f"Disagreements: {len(disagree)} / {len(comparison)}")
print(f"LLM correct in disagreements: {disagree['llm_correct'].sum()}")
print(f"XGBoost correct in disagreements: {disagree['xgb_correct'].sum()}")

if len(disagree) > 0:
    print("\nSample disagreements with LLM reasoning:")
    for _, row in disagree.head(5).iterrows():
        actual = 'Fully Paid' if row['actual'] == 1 else 'Charged Off'
        llm = 'Fully Paid' if row['llm_pred'] == 1 else 'Charged Off'
        xgb = 'Fully Paid' if row['xgb_pred'] == 1 else 'Charged Off'
        print(f"  Actual: {actual} | LLM: {llm} | XGBoost: {xgb}")
        print(f"  Reasoning: {row['llm_reasoning']}\n")

Disagreements: 16 / 100
LLM correct in disagreements: 4
XGBoost correct in disagreements: 12

Sample disagreements with LLM reasoning:
  Actual: Fully Paid | LLM: Fully Paid | XGBoost: Charged Off
  Reasoning: Despite high revolving utilization, the borrower has a very low debt-to-income ratio (19.96%), a long and clean credit history, and a verified income. The loan's purpose is to consolidate credit card debt to improve credit and reduce monthly payments, indicating a motivated borrower with a clear financial benefit from repayment.

  Actual: Fully Paid | LLM: Charged Off | XGBoost: Fully Paid
  Reasoning: The borrower exhibits significant financial strain with an extremely high revolving utilization rate of 79.7% and a large revolving balance relative to their low, unverified annual income. This combination suggests a high risk of default due to limited repayment capacity.

  Actual: Fully Paid | LLM: Charged Off | XGBoost: Fully Paid
  Reasoning: Despite a long credit history and 

## Export Results

In [10]:
os.makedirs(RESULTS_DIR, exist_ok=True)

comparison.to_csv(f"{RESULTS_DIR}/04b_llm_with_desc_results.csv", index=False)

# Load 04a metrics for side-by-side comparison
no_desc_metrics = pd.read_csv(f"{RESULTS_DIR}/04a_llm_no_desc_metrics.csv", index_col=0)

summary = pd.DataFrame([llm_metrics, xgb_metrics],
                        index=['LLM (With Desc)', 'XGBoost'])
all_results = pd.concat([no_desc_metrics.loc[['LLM (No Desc)']], summary])
all_results.to_csv(f"{RESULTS_DIR}/04b_llm_with_desc_metrics.csv")

print("All results comparison:")
print(all_results.to_string())

All results comparison:
                 accuracy  precision_charged_off  recall_charged_off  f1_charged_off  n_valid
LLM (No Desc)        0.56               0.208333              0.6250        0.312500      100
LLM (With Desc)      0.63               0.230769              0.5625        0.327273      100
XGBoost              0.71               0.290323              0.5625        0.382979      100
